# Carpet diagram — midHolocene temperature benchmarking

Step toward an update of IPCC AR5 WG1 Fig. 9.12 (a *portrait plot* / *carpet
diagram*). For each PMIP midHolocene (mid-Holocene (6 ka)) simulation we:

1. Load the annual-mean near-surface temperature field `tas_spatialmean_ann` from the
   CVDP output, and the matching `piControl` run.
2. Compute the **midHolocene − piControl anomaly** on each model's native grid.
3. **Sample** that anomaly field at the location of every proxy reconstruction.
4. Summarise the model–data mismatch as the **root-mean-squared error (RMSE)** against
   each reconstruction compilation (Bartlein (Bartlein et al. 2011), Temp12k (Kaufman et al. 2020), P2F
                      (Past 2 Future, in prep) and Erb (Holocene Reconstruction; Erb et al. 2022)).

Intermediate tables are written to `output/`. Run with the `my-cli-py` conda env.

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import xarray as xr

ROOT      = os.getcwd()  # run the notebook from the carpet_diagram/ directory
CVDP_DIR  = os.path.join(ROOT, 'cvdp_output_by_experiment')
EXPERIMENT = 'midHolocene'
RECON_DIR = os.path.join(ROOT, 'recons', EXPERIMENT)
OUT_DIR   = os.path.join(ROOT, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

VAR        = 'tas_spatialmean_ann'
print('experiment:', EXPERIMENT)
print('CVDP   :', CVDP_DIR)
print('recons :', RECON_DIR)
print('output :', OUT_DIR)

## Step 1 — Reconstruction compilations

Four midHolocene (6 ka) compilations. The first two are scattered proxy sites, each in
its own CSV with a plain header row and unit weight per point:

- **Bartlein** — the Bartlein et al. (2011) pollen-based mean-annual-temperature (MAT)
  anomalies; the annual anomaly is the `mat_anm_mean` column of a plain-header CSV.
- **Temp12k** — the Temp12k database extraction (Kaufman et al. 2020; described in
  Brierley et al. 2020); the annual anomaly is the `anom` column of a plain-header CSV.
- **P2F** — the Past 2 Future ocean-temperature compilation (Cutmore et al., Durham,
  in prep; work in progress), `P2F-recommended-SST.midHolocene.xlsx`. Only rows whose
  `Recommendation` column reads exactly `Recommend` are kept (others carry a note
  explaining why that site was excluded); the anomaly is `Annual SST` minus the site's
  `ERSSTv6 mean annual SST (1870-1899)` modern climatology.

Bartlein/Temp12k store latitude/longitude as plain `lat`/`lon` columns (longitudes
−180–180°); P2F uses `Latitude`/`Longitude`. All three are normalised to the shared
`compilation, reference, site, Proxy, Latitude, Longitude, Anom` schema used by the rest
of the pipeline.

The fourth is a gridded data assimilation product, flattened to points with a
`cos(latitude)` weight so its near-global RMSE is area-fair:

- **Erb** — the Holocene Reconstruction of Erb et al. (2022). Note the published
  `recon_tas_mean` field is referenced to **3–5 ka** ("The reference period of each
  reconstruction is 3-5 ka"), *not* to the pre-industrial, so it is not directly
  comparable with a model `midHolocene − piControl` anomaly. We therefore use
  `Erbetal2022_6ka_minus_0-1ka_anom.nc`, the 5.5–6.5 ka mean minus the 0–1 ka mean; the
  shared 3–5 ka baseline cancels in the difference, leaving a 6 ka − (0–1 ka) anomaly on
  the same footing as Osman's LGM field.

The fourth is a scattered-site marine compilation with unit weights:

- **P2F** — the recommended annual sea-surface-temperature records assembled for the
  Past2Future synthesis (`P2F_recommended_sst_midHolocene.csv`, a verbatim CSV dump of
  the supplied workbook). The `SST_Anomaly` column is each core's annual SST minus the
  ERSSTv6 1870–1899 mean at its grid cell, so it is already an anomaly. These are SSTs
  compared against model `tas`, on the same footing as the lig127k and midPliocene
  marine rows.

In [ ]:
# (compilation, filename, anomaly column, reference label, proxy label)
recon_specs = [
    ('Bartlein', 'Bartlein_mat.csv',       'mat_anm_mean', 'Bartlein et al. 2011',                  'pollen MAT'),
    ('Temp12k',  'temp12k_anom6k_mat.csv', 'anom',         'Temp12k (Kaufman et al. 2020)',         'multiproxy'),
]

frames = []
for comp, fname, anom_col, ref, proxy in recon_specs:
    df = pd.read_csv(os.path.join(RECON_DIR, fname))
    df.columns = [c.strip() for c in df.columns]  # some Bartlein headers carry leading spaces
    sub = pd.DataFrame({
        'compilation': comp,
        'reference': ref,
        'site': np.nan,          # these gridded compilations carry no site names
        'Proxy': proxy,
        'Latitude': df['lat'].astype(float),
        'Longitude': df['lon'].astype(float),
        'Anom': df[anom_col].astype(float),
    })
    sub['source_table'] = fname
    sub['weight'] = 1.0          # scattered sites: equal weight
    frames.append(sub)

# --- P2F: Past 2 Future SST compilation (Cutmore et al., in prep) ---
P2F_FILE = 'P2F-recommended-SST.midHolocene.xlsx'
p2f = pd.read_excel(os.path.join(RECON_DIR, P2F_FILE))
p2f = p2f[p2f['Recommendation'] == 'Recommend'].copy()
p2f_df = pd.DataFrame({
    'compilation': 'P2F', 'reference': 'Past 2 Future (Cutmore et al., in prep)',
    'site': p2f['Core'].astype(str).str.strip(),
    'Proxy': p2f['Proxy/Proxies'],
    'Latitude': p2f['Latitude'].astype(float), 'Longitude': p2f['Longitude'].astype(float),
    'Anom': (p2f['Annual SST'] - p2f['ERSSTv6 mean annual SST (1870-1899)']).astype(float),
})
p2f_df['source_table'] = P2F_FILE
p2f_df['weight'] = 1.0            # scattered sites: equal weight
frames.append(p2f_df)

# --- Erb: flatten the gridded 6 ka - (0-1 ka) reconstruction to points ---
ERB_FILE = 'Erbetal2022_6ka_minus_0-1ka_anom.nc'
ds = xr.open_dataset(os.path.join(RECON_DIR, ERB_FILE), decode_times=False)
tas = ds['recon_tas_mean'].where(np.abs(ds['recon_tas_mean']) < 1e30)
lon2d, lat2d = np.meshgrid(tas['lon'].values, tas['lat'].values)
erb = pd.DataFrame({
    'compilation': 'Erb', 'reference': 'Erb et al. 2022', 'site': np.nan,
    'Proxy': 'Holocene Recon. SAT (assim.)',
    'Latitude': lat2d.ravel().astype(float), 'Longitude': lon2d.ravel().astype(float),
    'Anom': tas.values.ravel().astype(float),
})
erb['source_table'] = ERB_FILE
# Regular lat/lon grid: cos-latitude weight makes the RMSE area-fair.
erb['weight'] = np.cos(np.deg2rad(erb['Latitude']))
frames.append(erb)

# --- P2F: the recommended annual SST records (scattered marine cores) ---
# A verbatim CSV dump of the P2F workbook (see recons/README for the command).
# `SST_Anomaly` is the core's annual SST minus the ERSSTv6 1870-1899 mean at its
# grid cell, so it is already the anomaly we want and needs no re-referencing.
# These are SSTs benchmarked against model `tas`, exactly as the lig127k Hoffman
# and Capron and the midPliocene Foley-Dowsett rows already are.
P2F_FILE = 'P2F_recommended_sst_midHolocene.csv'
p2f = pd.read_csv(os.path.join(RECON_DIR, P2F_FILE))
p2f.columns = [c.strip() for c in p2f.columns]

# Coordinate check: the site `Latitude` should agree with the spreadsheet's own
# ERSST grid latitude. Where the two disagree in sign the site column has a sign
# typo (e.g. GeoB7112-5, a Chile-margin core listed at +24 deg N), so take the
# hemisphere from the ERSST cell. Rows within 2 deg of the equator are skipped:
# there a sign difference is just rounding across it.
flip = (np.sign(p2f['Latitude']) != np.sign(p2f['ERSST Lat'])) & (p2f['ERSST Lat'].abs() > 2)
if flip.any():
    print('P2F: latitude sign taken from the ERSST cell for',
          ', '.join(p2f.loc[flip, 'Core'].astype(str)))
    p2f.loc[flip, 'Latitude'] = -p2f.loc[flip, 'Latitude']

p2f_pts = pd.DataFrame({
    'compilation': 'P2F', 'reference': 'P2F recommended SST records',
    'site': p2f['Core'].astype(str),
    'Proxy': p2f['Proxy/Proxies'].astype(str),
    'Latitude': p2f['Latitude'].astype(float),
    'Longitude': p2f['Longitude'].astype(float),
    'Anom': p2f['SST_Anomaly'].astype(float),
})
p2f_pts['source_table'] = P2F_FILE
p2f_pts['weight'] = 1.0                   # scattered sites: equal weight
frames.append(p2f_pts)

recon = pd.concat(frames, ignore_index=True)
recon = recon.dropna(subset=['Latitude', 'Longitude', 'Anom'])

print(recon.groupby('compilation').size())
recon_out = os.path.join(OUT_DIR, f'recon_points_{EXPERIMENT}.csv')
recon.to_csv(recon_out, index=False)
print('wrote', recon_out)
recon.head()

## Step 2 — Model midHolocene − piControl anomalies

Each model writes its CVDP field on its own native grid, so anomalies are computed
per model. We pair every `midHolocene` file with the same model's `piControl` file
(the main file is the one named `<model>_<experiment>.cvdp_data.<years>.nc`; auxiliary
per-variable files such as `.siconc.` / `.zos.` / `.monsoon.` / `.tas.indices.` carry a
extra token before the years and are skipped). If the two grids ever differ, the control
is bilinearly regridded onto the midHolocene grid before differencing.

In [ ]:
def model_files(experiment):
    """Map model name -> path for the main CVDP file of an experiment.

    Only `<model>_<experiment>.cvdp_data.<start>-<end>.nc` is the main file; the
    auxiliary per-variable outputs (`.siconc.`, `.zos.`, `.monsoon.`, `.tas.indices.`)
    put an extra token before the year range and hold no `tas_spatialmean_ann`.
    """
    out = {}
    pat = os.path.join(CVDP_DIR, experiment, f'*_{experiment}.cvdp_data.*.nc')
    main = re.compile(rf'^(?P<model>.+)_{re.escape(experiment)}\.cvdp_data\.\d+-\d+\.nc$')
    for f in sorted(glob.glob(pat)):
        m = main.match(os.path.basename(f))
        if m:
            out[m.group('model')] = f
    return out

def has_var(path):
    """True if the CVDP file actually carries the tas field (a few don't)."""
    with xr.open_dataset(path, decode_times=False) as ds:
        return VAR in ds.variables

exp_files = model_files(EXPERIMENT)
pi_files  = model_files('piControl')
paired = sorted(set(exp_files) & set(pi_files))
# Some CVDP files lack tas_spatialmean_ann entirely — drop those models.
models = [m for m in paired if has_var(exp_files[m]) and has_var(pi_files[m])]
print(f'{len(models)} models with both {EXPERIMENT} and piControl (and a tas field):')
print(models)
missing_pi = sorted(set(exp_files) - set(pi_files))
if missing_pi:
    print(f'{EXPERIMENT} models with no piControl (skipped):', missing_pi)
no_var = [m for m in paired if m not in models]
if no_var:
    print(f'models dropped — no {VAR} in CVDP file:', no_var)

In [ ]:
def load_field(path):
    da = xr.open_dataset(path, decode_times=False)[VAR].sortby('lat').sortby('lon')
    # A few CVDP grids (e.g. LOVECLIM piControl) carry duplicate lon values,
    # which break interpolation; keep the first occurrence of each coordinate.
    for dim in ('lat', 'lon'):
        _, idx = np.unique(da[dim].values, return_index=True)
        if len(idx) != da.sizes[dim]:
            da = da.isel({dim: np.sort(idx)})
    return da

anomalies = {}
for m in models:
    exp_field = load_field(exp_files[m])
    pi  = load_field(pi_files[m])
    if exp_field.shape != pi.shape or not (np.allclose(exp_field.lat, pi.lat) and np.allclose(exp_field.lon, pi.lon)):
        pi = pi.interp(lat=exp_field.lat, lon=exp_field.lon)
    anomalies[m] = (exp_field - pi).rename('tas_anom')
    print(f'{m:18s} grid {exp_field.shape}  mean anom {float(anomalies[m].mean()):+.2f} C')

## Step 3 — Sample model anomalies at reconstruction locations

Model longitudes run 0–360°, the proxy longitudes −180–180°, so targets are wrapped to
0–360 and the field is made cyclic in longitude before bilinear interpolation.

Each recon point carries a `weight` used later in the RMSE. Scattered-site compilations
weight every point equally (1.0); gridded near-global reconstructions (Cleator, Osman,
Erb, Tierney) set `weight = cos(latitude)` so the RMSE is area-fair rather than
pole-heavy.

In [ ]:
def sample_points(field, lats, lons):
    """Bilinearly sample a (lat, lon) field at scattered points; lon made cyclic."""
    lon_cyc = np.append(field.lon.values, field.lon.values[0] + 360.0)
    fcyc = xr.concat([field, field.isel(lon=0)], dim='lon').assign_coords(lon=lon_cyc)
    ta = xr.DataArray(np.asarray(lats), dims='point')
    to = xr.DataArray(np.asarray(lons) % 360.0, dims='point')
    return fcyc.interp(lat=ta, lon=to).values

sampled = recon[['compilation', 'reference', 'site', 'Proxy',
                 'Latitude', 'Longitude', 'Anom']].copy()
sampled = sampled.rename(columns={'Anom': 'recon_anom'})
# Optional per-point weight (defaults to equal weighting when Step 1 omits it).
sampled['weight'] = recon['weight'].values if 'weight' in recon.columns else 1.0
for m in models:
    sampled[m] = sample_points(anomalies[m], sampled['Latitude'].values, sampled['Longitude'].values)

sampled_out = os.path.join(OUT_DIR, f'model_anom_at_recon_{EXPERIMENT}.csv')
sampled.to_csv(sampled_out, index=False)
print('wrote', sampled_out, '  shape', sampled.shape)
sampled.head()

## Step 4 — RMSE of each model against each compilation

For every model × compilation we take the model-minus-proxy difference across all proxy
points in that compilation and report the (weight-weighted) RMSE and bias, plus the number
of points contributing (a point off the model grid edge can return NaN). With unit weights
this is the ordinary RMSE; gridded compilations use `cos(latitude)` weights (Step 3).

In [ ]:
records = []
for m in models:
    for comp, grp in sampled.groupby('compilation'):
        diff = grp[m].values - grp['recon_anom'].values
        w = grp['weight'].values
        valid = np.isfinite(diff) & np.isfinite(w)
        n = int(valid.sum())
        if n:
            dv, wv = diff[valid], w[valid]
            rmse = float(np.sqrt(np.sum(wv * dv ** 2) / np.sum(wv)))
            bias = float(np.sum(wv * dv) / np.sum(wv))
        else:
            rmse = bias = np.nan
        records.append({'model': m, 'compilation': comp, 'n_points': n,
                        'rmse': rmse, 'bias': bias})

rmse_long = pd.DataFrame(records)
rmse_wide = rmse_long.pivot(index='model', columns='compilation', values='rmse')
rmse_wide.columns = [f'{c}_RMSE' for c in rmse_wide.columns]

rmse_long.to_csv(os.path.join(OUT_DIR, f'rmse_long_{EXPERIMENT}.csv'), index=False)
rmse_wide.to_csv(os.path.join(OUT_DIR, f'rmse_summary_{EXPERIMENT}.csv'))
print(f'wrote rmse_long_{EXPERIMENT}.csv and rmse_summary_{EXPERIMENT}.csv')
rmse_wide.sort_values(rmse_wide.columns[0])

These RMSE values are the building blocks of the carpet diagram: one column per model,
one row per (period, reconstruction compilation), coloured by RMSE. `carpet_figure.py`
reads every `output/rmse_long_<period>.csv`, so re-running this notebook for a new period
makes its rows appear in the portrait plot automatically.